# modification
1) unfreeze last 2 blocks other than FCL
2) fedprox instead of fed avg
additonally

1) unfreezing 6th layer
2) added class weights balancer in the loss
3) added cosine scheduler
4) the lr is set by the server side optimizer so it will change only after the rounds not within each epochs so modifiying it to local optimizer



In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
import os
import copy
from collections import OrderedDict, Counter
import numpy as np
from collections import OrderedDict
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
from collections import defaultdict
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.metrics import classification_report

In [5]:
class Net(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        # Load the base EfficientNet model with pre-trained weights
        self.base = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)

        # 1. Freeze the entire backbone initially
        for p in self.base.parameters():
            p.requires_grad = False

        # 2. PARTIAL UNFREEZE: Unfreeze the last two blocks (features.7 and features.8)
        # EfficientNet features are in blocks (0 to 8). We unfreeze the last few.
        # Unfreeze features.7 (Block 7)
        for p in self.base.features[6].parameters():
            p.requires_grad = True

        for p in self.base.features[7].parameters():
            p.requires_grad = True

        # Unfreeze features.8 (Block 8, which contains the final convolution layer)
        for p in self.base.features[8].parameters():
            p.requires_grad = True

        # 3. Replace and Unfreeze the final fully connected (FC) classifier
        num_ftrs = self.base.classifier[-1].in_features
        self.base.classifier[-1] = nn.Linear(num_ftrs, num_classes)

        for p in self.base.classifier.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.base(x)

In [6]:
img_tf = Compose([
    Resize((256, 256)),
    ToTensor(),
    Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [7]:
DATA_ROOT = "/kaggle/input/distributed-dataset/Splitted_data"
BATCH_SIZE = 32
N_FOLDS = 5
SEED = 42

def load_client_data(client_id):
    path = os.path.join(DATA_ROOT, f"Client_{client_id}")
    dataset = ImageFolder(path, transform=img_tf)

    n = len(dataset)
    t = int(0.8 * n)
    v = n - t

    train_ds, val_ds = random_split(dataset, [t, v], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)
    return train_loader, val_loader

def get_client_dataset(client_id):
    path = os.path.join(DATA_ROOT, f"Client_{client_id}")
    dataset = ImageFolder(path, transform=img_tf)
    return dataset

In [8]:
def get_stratified_folds(dataset, n_splits=N_FOLDS, seed=SEED):
    # ImageFolder stores labels as dataset.targets on some torchvision versions, otherwise extract
    if hasattr(dataset, "targets"):
        labels = np.array(dataset.targets)
    else:
        labels = np.array([dataset[i][1] for i in range(len(dataset))])

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = list(skf.split(np.zeros(len(labels)), labels))
    # folds is list of (train_idx, val_idx)
    return folds

In [9]:
def average_state_dicts(state_dicts):
    avg = copy.deepcopy(state_dicts[0])
    n = len(state_dicts)
    for k in avg.keys():
        # accumulate
        for i in range(1, n):
            avg[k] = avg[k] + state_dicts[i][k]
        avg[k] = avg[k] / n
    return avg

In [10]:
def local_train(model, loader, epochs, lr, device):
    model.to(device)
    model.train()

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()

    return model.state_dict()  # return updated weights

In [11]:
from sklearn.metrics import classification_report

def local_eval(model, data_loader, device,class_weights):
    model.eval()
    model.to(device)
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)

    # Generate a detailed classification report
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)

    return avg_loss, report

In [12]:
def local_train_fedprox(model, loader, epochs, lr, mu, device, class_weights):
    model.to(device)
    model.train()

    # Deepcopy the global model weights w^t for the proximal term calculation
    w_global = OrderedDict(copy.deepcopy(model.state_dict()))

    # 1. Instantiate Optimizer
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # 2. Instantiate Local Scheduler
    # T_max is set to the number of local epochs for decay across the local training session
    local_scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-7)

    # 3. Use the provided class_weights in CrossEntropyLoss
    crit = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()

            # --- A. Standard Loss (Cross-Entropy) ---
            loss = crit(model(x), y)

            # --- B. Proximal Term (FedProx) ---
            proximal_term = 0.0

            # Iterate through all local model parameters (w)
            for name, param in model.named_parameters():
                if name in w_global:
                    # Calculate ||w - w^t||^2 for each layer
                    # Note: We must move w_global[name] to the current device
                    # before calculating the difference.
                    proximal_term += torch.sum((param - w_global[name].to(device)) ** 2)

            # Add the FedProx term: Loss + (mu/2) * ||w - w^t||^2
            loss += (mu / 2.0) * proximal_term

            # --- C. Backpropagation ---
            loss.backward()
            opt.step()

        # 4. Step the Local Scheduler after each epoch
        # This will adjust the LR for the next epoch's optimization steps.
        local_scheduler.step()
        
        # Optional: Check the decay
        # print(f"  Epoch {epoch+1}/{epochs}: Local LR={local_scheduler.get_last_lr()[0]:.8f}")

    return model.state_dict() # return updated weights

In [13]:
def fed_avg(models):
    avg = copy.deepcopy(models[0])
    for k in avg.keys():
        for i in range(1, len(models)):
            avg[k] += models[i][k]
        avg[k] = avg[k] / len(models)
    return avg

In [14]:
metrics_history = defaultdict(lambda: defaultdict(list))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLIENTS = 3
ROUNDS = 10
LOCAL_EPOCHS = 10
LR = 0.0001
MU = 0.01
NUM_CLASSES = 6
N_FOLDS = 5
SEED = 42
BATCH_SIZE = 32  # make sure this is defined

global_model = Net()

# Load all client validation data once
client_val_loaders = {}
for cid in range(1, NUM_CLIENTS + 1):
    _, val_loader = load_client_data(cid)
    client_val_loaders[cid] = val_loader
    
client_datasets = {}
client_folds = {}

for cid in range(1, NUM_CLIENTS + 1):
    ds = get_client_dataset(cid)
    client_datasets[cid] = ds
    folds = get_stratified_folds(ds, n_splits=N_FOLDS, seed=SEED)
    client_folds[cid] = folds
    print(f"Client {cid}: {len(ds)} samples, {len(folds)} folds created.")

# Compute class weights from indices
def compute_class_weights_from_indices(dataset, indices, num_classes=NUM_CLASSES, device=DEVICE):
    counts = [0] * num_classes
    for i in indices:
        _, lbl = dataset[i]
        counts[lbl] += 1
    max_count = max(c for c in counts if c > 0) if any(c>0 for c in counts) else 1
    weights = [(max_count / c) if c>0 else 0.0 for c in counts]
    return torch.tensor(weights, dtype=torch.float32).to(device)

# Function to print aligned metrics
def print_aligned_report(report, title):
    print(f"\n--- {title} ---")
    header = f"{'class':<10}{'precision':<12}{'recall':<10}{'f1-score':<12}{'support':<10}"
    print(header)
    print("-" * len(header))
    for label in sorted([k for k in report.keys() if k.isdigit()]):
        metrics = report[label]
        print(f"{label:<10}   {metrics['precision']:.4f}   {metrics['recall']:.4f}    {metrics['f1-score']:.4f}     {metrics['support']:<10}")
    print("-" * len(header))
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    total_support = int(weighted_avg['support'])
    print(f"{'accuracy':<10}{'':<12}{'':<10}{report['accuracy']:.4f}{total_support:<10}")

# Compute global class counts
from collections import Counter
global_class_counts = Counter()
for cid in range(1, NUM_CLIENTS + 1):
    train_loader, _ = load_client_data(cid)
    for _, labels in train_loader:
        global_class_counts.update(labels.tolist())

class_counts = [global_class_counts[i] for i in range(NUM_CLASSES)]
print(f"Global Training Class Counts: {class_counts}")
max_count = max(c for c in class_counts if c > 0)
class_weights = torch.tensor([max_count / c if c > 0 else 0 for c in class_counts], dtype=torch.float32).to(DEVICE)

# Track best metrics
best_global_accuracy = -1.0
best_metrics_report = None
best_round_info = {"round": -1, "client_id": -1, "fold": -1}

# Initialize metrics history
metrics_history = {r: {} for r in range(ROUNDS)}

print("Starting FL Training with per-client 5-fold training and aggregation...")

for r in range(ROUNDS):
    print(f"\n----- Round {r+1} (Training across clients with 5-fold per-client) -----")
    current_lr = LR
    print(f"Current Learning Rate: {current_lr:.8f}")

    client_weights_for_aggregation = []

    for cid in range(1, NUM_CLIENTS + 1):
        ds = client_datasets[cid]
        folds = client_folds[cid]
        per_fold_state_dicts = []
        print(f"\n Client {cid}: running {N_FOLDS} local trainings (one per fold)...")

        for f_idx, (train_idx, val_idx) in enumerate(folds):
            train_subset = Subset(ds, train_idx)
            val_subset = Subset(ds, val_idx)
            train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

            cw = compute_class_weights_from_indices(ds, train_idx, num_classes=NUM_CLASSES, device=DEVICE)

            local_model = copy.deepcopy(global_model)
            updated_state = local_train_fedprox(local_model, train_loader, LOCAL_EPOCHS, current_lr, MU, DEVICE, cw)
            per_fold_state_dicts.append(copy.deepcopy(updated_state))

            local_model.load_state_dict(updated_state)
            val_loss, val_report = local_eval(local_model, val_loader, DEVICE, cw)
            if (cid, f_idx) not in metrics_history[r]:
                metrics_history[r][(cid, f_idx)] = []
            metrics_history[r][(cid, f_idx)].append({"loss": val_loss, "report": val_report})

            acc = val_report.get("accuracy", 0.0)
            print(f"   Client {cid} Fold {f_idx+1}/{N_FOLDS}: Val Loss: {val_loss:.4f}, Val Acc: {acc:.4f}")

            if acc > best_global_accuracy:
                best_global_accuracy = acc
                best_metrics_report = copy.deepcopy(val_report)
                best_round_info = {"round": r+1, "client_id": cid, "fold": f_idx+1}

        # Average fold weights for this client
        client_agg_state = average_state_dicts(per_fold_state_dicts)
        client_weights_for_aggregation.append(client_agg_state)
        print(f" Client {cid}: averaged {len(per_fold_state_dicts)} fold-updates into a single client update.")

    # Server aggregation
    new_global_weights = fed_avg(client_weights_for_aggregation)
    global_model.load_state_dict(new_global_weights)
    print("Server aggregated weights (FedAvg of client-aggregated updates).")

    # Round summary with all metrics
    print(f"\n----- Round {r+1} (Summary of per-client fold validations) -----")
    for cid in range(1, NUM_CLIENTS + 1):
        fold_reports = [metrics_history[r][(cid, f_idx)][-1]["report"] for f_idx in range(N_FOLDS)]
        fold_accs = [r["accuracy"] for r in fold_reports]
        mean_acc = np.mean(fold_accs)
        std_acc = np.std(fold_accs)
        print(f"\nClient {cid}: Fold Accuracies mean: {mean_acc:.4f}, std: {std_acc:.4f}")
        for f_idx, report in enumerate(fold_reports):
            print_aligned_report(report, f"Client {cid} Fold {f_idx+1} Metrics (Round {r+1})")

    print("\n" + "="*50)

print("\nFL Training Completed.\n")
print("="*50)
print("             BEST LOCAL-FOLD METRICS FOUND            ")
print(f" (Achieved in Round {best_round_info['round']}, on Client {best_round_info['client_id']}, Fold {best_round_info['fold']})")
print("="*50)

if best_metrics_report:
    print_aligned_report(best_metrics_report, "Best Fold-wise Class-wise Performance")
    print(f"\nBest Validation Accuracy (fold-level): {best_global_accuracy:.4f}")
else:
    print("No validation results recorded.")


print("\n===== Final Evaluation on Each Client =====")
for cid in range(1, NUM_CLIENTS + 1):
    _, val_loader = load_client_data(cid)
    loss, report = local_eval(global_model, val_loader, DEVICE, class_weights)
    print(f"\nClient {cid} Validation:")
    print("Loss:", loss)
    print("Accuracy:", report["accuracy"])
    print_aligned_report(report, f"Client {cid} Final Metrics")


Client 1: 1420 samples, 5 folds created.
Client 2: 1395 samples, 5 folds created.
Client 3: 1387 samples, 5 folds created.
Global Training Class Counts: [361, 104, 1890, 419, 373, 214]
Starting FL Training with per-client 5-fold training and aggregation...

----- Round 1 (Training across clients with 5-fold per-client) -----
Current Learning Rate: 0.00010000

 Client 1: running 5 local trainings (one per fold)...
   Client 1 Fold 1/5: Val Loss: 0.7579, Val Acc: 0.7183
   Client 1 Fold 2/5: Val Loss: 0.7932, Val Acc: 0.7465
   Client 1 Fold 3/5: Val Loss: 0.8217, Val Acc: 0.7359
   Client 1 Fold 4/5: Val Loss: 0.7984, Val Acc: 0.7289
   Client 1 Fold 5/5: Val Loss: 0.7585, Val Acc: 0.7676
 Client 1: averaged 5 fold-updates into a single client update.

 Client 2: running 5 local trainings (one per fold)...
   Client 2 Fold 1/5: Val Loss: 0.8229, Val Acc: 0.7455
   Client 2 Fold 2/5: Val Loss: 0.7393, Val Acc: 0.7921
   Client 2 Fold 3/5: Val Loss: 0.7702, Val Acc: 0.7634
   Client 2 Fol

In [ ]:
print("\n===== Final Evaluation on Each Client =====")
for cid in range(1, NUM_CLIENTS + 1):
    _, val_loader = load_client_data(cid)
    loss, report = local_eval(global_model, val_loader, DEVICE,class_weights)
    print(f"\nClient {cid} Validation:")
    print("Loss:", loss)
    print("Accuracy:", report["accuracy"])
    print(report["macro avg"])